# Original IP102 ResNet50 Classification Report

This notebook downloads and evaluates the original **IP102 insect pest dataset** from Kaggle dataset `rt1mhjbn/ip102-dataset`.

It will generate a real 102-class classification report with precision, recall, F1-score, support, accuracy, macro average, and weighted average. Kaggle API authentication may be required on the first download.

In [1]:
from pathlib import Path
import json
import csv
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import torchvision.transforms as transforms

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_ROOT = ROOT / 'data'
CHECKPOINT = ROOT / 'resnet50_0.497.pkl'
OUTPUT_CSV = ROOT / 'training_output' / 'ip102_predictions.csv'
OUTPUT_JSON = ROOT / 'training_output' / 'ip102_classification_report.json'

with (ROOT / 'public' / 'pest_model_metadata.json').open(encoding='utf-8') as file:
    metadata = json.load(file)

CLASS_NAMES = metadata['classes']
NUM_CLASSES = metadata['num_classes']
IMAGE_SIZE = metadata['image_size']

if torch.cuda.is_available():
    DEVICE = torch.device('cuda:0')
    GPU_NAME = torch.cuda.get_device_name(DEVICE)
    GPU_MEMORY_GB = torch.cuda.get_device_properties(DEVICE).total_memory / (1024 ** 3)
    print(f'Using GPU: {GPU_NAME} ({GPU_MEMORY_GB:.2f} GB)')
else:
    DEVICE = torch.device('cpu')
    GPU_NAME = 'CPU kernel - CUDA unavailable'
    print('WARNING: This notebook kernel cannot see the RTX 3050.')
    print('Select the Anaconda/Python environment where torch.cuda.is_available() is True to use the GPU.')

print(f'Device: {DEVICE}')
print(f'Classes: {NUM_CLASSES}')
print(f'IP102 data root: {DATA_ROOT}')

Using GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU (6.00 GB)
Device: cuda:0
Classes: 102
IP102 data root: c:\Users\sunet\Documents\SIH2026\KR_AI\KrishirakshaAI\data


In [3]:
TEST_ANNOTATIONS = DATA_ROOT / 'test.txt'
IMAGE_ROOT = DATA_ROOT / 'classification' / 'test'

if not TEST_ANNOTATIONS.exists():
    raise FileNotFoundError(f'IP102 test annotations not found: {TEST_ANNOTATIONS}')
if not IMAGE_ROOT.exists():
    raise FileNotFoundError(f'IP102 test image folder not found: {IMAGE_ROOT}')

image_index = {
    image_path.name: image_path
    for image_path in IMAGE_ROOT.rglob('*')
    if image_path.is_file()
}

records = []
with TEST_ANNOTATIONS.open(encoding='utf-8') as file:
    for line_number, line in enumerate(file, start=1):
        line = line.strip()
        if not line:
            continue
        image_name, label_text = line.rsplit(maxsplit=1)
        label = int(label_text)
        if not 0 <= label < NUM_CLASSES:
            raise ValueError(f'Invalid IP102 class {label} on line {line_number}')
        image_path = image_index.get(Path(image_name).name)
        if image_path is None:
            raise FileNotFoundError(f'IP102 image not found: {image_name}')
        records.append((image_path, label))

print(f'Loaded {len(records):,} original IP102 test records')

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=metadata['normalization']['mean'],
        std=metadata['normalization']['std'],
    ),
])

class IP102TestDataset(Dataset):
    def __init__(self, entries):
        self.entries = entries

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, index):
        image_path, label = self.entries[index]
        image = Image.open(image_path).convert('RGB')
        return val_transform(image), label

test_loader = DataLoader(
    IP102TestDataset(records),
    batch_size=128,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

Loaded 22,619 original IP102 test records


In [4]:
if not CHECKPOINT.exists():
    raise FileNotFoundError(f'ResNet50 checkpoint not found: {CHECKPOINT}')

model = models.resnet50(weights=None)
model.fc = nn.Linear(2048, NUM_CLASSES)
state_dict = torch.load(CHECKPOINT, map_location='cpu', weights_only=False)
model.load_state_dict(state_dict, strict=True)
model.to(DEVICE)
model.eval()

true_labels = []
predicted_labels = []

with torch.inference_mode():
    for images, labels in test_loader:
        images = images.to(DEVICE, non_blocking=DEVICE.type == 'cuda')
        if DEVICE.type == 'cuda':
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                logits = model(images)
        else:
            logits = model(images)
        predictions = logits.argmax(dim=1).cpu().tolist()
        predicted_labels.extend(predictions)
        true_labels.extend(labels.tolist())

accuracy = float(np.mean(np.array(true_labels) == np.array(predicted_labels)))
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
with OUTPUT_CSV.open('w', newline='', encoding='utf-8') as file:
    writer = csv.writer(file)
    writer.writerow(['true_label', 'predicted_label'])
    writer.writerows(zip(true_labels, predicted_labels))

if DEVICE.type == 'cuda':
    torch.cuda.synchronize()
print(f'Test accuracy: {accuracy * 100:.2f}%')
print(f'Inference device: {DEVICE} ({GPU_NAME})')
print(f'Predictions saved to: {OUTPUT_CSV}')

Test accuracy: 50.32%
Inference device: cuda:0 (NVIDIA GeForce RTX 3050 6GB Laptop GPU)
Predictions saved to: c:\Users\sunet\Documents\SIH2026\KR_AI\KrishirakshaAI\training_output\ip102_predictions.csv


In [5]:
from sklearn.metrics import classification_report

report = classification_report(
    true_labels,
    predicted_labels,
    labels=list(range(NUM_CLASSES)),
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0,
)

report_table = pd.DataFrame(report).T.rename(columns={'f1-score': 'f1_score'})
display(report_table.style.format({
    'precision': '{:.4f}',
    'recall': '{:.4f}',
    'f1_score': '{:.4f}',
    'support': '{:,.0f}',
}))

weighted_f1 = report['weighted avg']['f1-score']
macro_f1 = report['macro avg']['f1-score']
print(f'Primary metric - weighted F1: {weighted_f1 * 100:.2f}%')
print(f'Macro F1: {macro_f1 * 100:.2f}%')
print(f'Accuracy: {report["accuracy"] * 100:.2f}%')

rare_class_report = report_table.loc[CLASS_NAMES].copy()
rare_class_report['class'] = rare_class_report.index
rare_class_report = rare_class_report.sort_values(
    ['support', 'f1_score'], ascending=[True, True]
)
print('Rare / difficult classes')
display(rare_class_report[['class', 'support', 'precision', 'recall', 'f1_score']].head(20).style.format({
    'precision': '{:.4f}',
    'recall': '{:.4f}',
    'f1_score': '{:.4f}',
    'support': '{:,.0f}',
}))

with OUTPUT_JSON.open('w', encoding='utf-8') as file:
    json.dump(report, file, indent=2)

print(f'Classification report saved to: {OUTPUT_JSON}')

,precision,recall,f1_score,support
rice leaf roller,0.5271,0.6955,0.5997,335
rice leaf caterpillar,0.2586,0.2041,0.2281,147
paddy stem maggot,0.1505,0.3544,0.2113,79
asiatic rice borer,0.4838,0.4241,0.4519,316
yellow rice borer,0.7826,0.1184,0.2057,152
rice gall midge,0.7473,0.4474,0.5597,152
Rice Stemfly,0.2857,0.4144,0.3382,111
brown plant hopper,0.3352,0.4741,0.3927,251
white backed plant hopper,0.2844,0.3545,0.3156,268
small brown plant hopper,0.4615,0.0723,0.1250,166


Primary metric - weighted F1: 48.97%
Macro F1: 41.40%
Accuracy: 50.32%
Rare / difficult classes


,class,support,precision,recall,f1_score
Erythroneura apicalis,Erythroneura apicalis,22,0.6250,0.2273,0.3333
Parlatoria zizyphus Lucus,Parlatoria zizyphus Lucus,23,0.0000,0.0000,0.0000
Brevipoalpus lewisi McGregor,Brevipoalpus lewisi McGregor,24,0.6522,0.6250,0.6383
Polyphagotars onemus latus,Polyphagotars onemus latus,26,0.5000,0.1923,0.2778
Mango flat beak leafhopper,Mango flat beak leafhopper,28,0.6667,0.0714,0.1290
Nipaecoccus vastalor,Nipaecoccus vastalor,30,0.3488,0.5000,0.4110
beet fly,beet fly,32,0.2000,0.0938,0.1277
cerodonta denticornis,cerodonta denticornis,42,0.2963,0.1905,0.2319
parathrene regalis,parathrene regalis,42,0.6957,0.3810,0.4923
white margined moth,white margined moth,45,0.0000,0.0000,0.0000


Classification report saved to: c:\Users\sunet\Documents\SIH2026\KR_AI\KrishirakshaAI\training_output\ip102_classification_report.json


## Ensemble and Long-Tail Evaluation

The ensemble cell combines class-probability files from independently trained ResNet50, EfficientNet, and ViT models. It selects the ensemble by weighted F1, not accuracy. Each file must have shape `(number_of_test_images, 102)` and use the same IP102 test ordering.

In [7]:
from sklearn.metrics import classification_report

probability_files = {
    'ResNet50': ROOT / 'training_output' / 'ip102_probs_resnet50.npy',
    'EfficientNet': ROOT / 'training_output' / 'ip102_probs_efficientnet.npy',
    'ViT': ROOT / 'training_output' / 'ip102_probs_vit.npy',
}
available_probabilities = {
    name: np.load(path)
    for name, path in probability_files.items()
    if path.exists()
}

if len(available_probabilities) < 2:
    print('Ensemble not evaluated yet.')
    print('Train at least two models and save test probabilities to:')
    for path in probability_files.values():
        print(f'  {path}')
else:
    probability_shapes = {name: values.shape for name, values in available_probabilities.items()}
    if any(shape != (len(true_labels), NUM_CLASSES) for shape in probability_shapes.values()):
        raise ValueError(f'Probability arrays must have shape ({len(true_labels)}, {NUM_CLASSES}); got {probability_shapes}')

    ensemble_probabilities = np.mean(list(available_probabilities.values()), axis=0)
    ensemble_predictions = ensemble_probabilities.argmax(axis=1)
    ensemble_report = classification_report(
        true_labels,
        ensemble_predictions,
        labels=list(range(NUM_CLASSES)),
        target_names=CLASS_NAMES,
        output_dict=True,
        zero_division=0,
    )
    print(f'Ensemble models: {", ".join(available_probabilities)}')
    print(f'Ensemble accuracy: {ensemble_report["accuracy"] * 100:.2f}%')
    print(f'Ensemble weighted F1: {ensemble_report["weighted avg"]["f1-score"] * 100:.2f}%')
    print(f'Ensemble macro F1: {ensemble_report["macro avg"]["f1-score"] * 100:.2f}%')
    display(pd.DataFrame(ensemble_report).T.style.format({
        'precision': '{:.4f}',
        'recall': '{:.4f}',
        'f1-score': '{:.4f}',
        'support': '{:,.0f}',
    }))

Ensemble models: ResNet50, EfficientNet, ViT
Ensemble accuracy: 50.30%
Ensemble weighted F1: 48.95%
Ensemble macro F1: 41.38%


,precision,recall,f1-score,support
rice leaf roller,0.5261,0.6925,0.5979,335
rice leaf caterpillar,0.2542,0.2041,0.2264,147
paddy stem maggot,0.1497,0.3544,0.2105,79
asiatic rice borer,0.4838,0.4241,0.4519,316
yellow rice borer,0.7826,0.1184,0.2057,152
rice gall midge,0.7556,0.4474,0.5620,152
Rice Stemfly,0.2840,0.4144,0.3370,111
brown plant hopper,0.3333,0.4622,0.3873,251
white backed plant hopper,0.2824,0.3657,0.3187,268
small brown plant hopper,0.4583,0.0663,0.1158,166
